<a href="https://colab.research.google.com/github/vishnu1234555/ML/blob/main/IMBD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [2]:
import pandas as pd
import re
import os

# 1. Locate and Load the Data
dataset_dir = "/kaggle/input/imdb-dataset-of-50k-movie-reviews"
# Dynamically grab the CSV to avoid capitalization typos (usually 'IMDB Dataset.csv')
csv_file = [f for f in os.listdir(dataset_dir) if f.endswith('.csv')][0]
file_path = os.path.join(dataset_dir, csv_file)

print(f"Loading data from: {file_path}")
data= pd.read_csv(file_path)

Loading data from: /kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [3]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Clean Text and Encode Labels (Chained Pandas Operations)
data['clean_review'] = data['review'].str.replace(r'<br\s*/?>', ' ', regex=True)
data['clean_review'] = data['clean_review'].str.replace(r'[^a-zA-Z\s]', '', regex=True).str.lower().str.strip()
data['label'] = data['sentiment'].map({'positive': 1, 'negative': 0})

# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    data['clean_review'], data['label'], test_size=0.2, random_state=42
)

# 3. Vectorize (The Matrix Conversion)
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

1. Cleaning text pipeline running...
2. Vectorizing text into sparse matrices...
Done. Your training text is now a matrix of shape: (40000, 5000)


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# 2. Check basic accuracy
predictions = model.predict(X_test_vec)
print(f"Model Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")

# 3. Test your own review
custom_review = "This movie was an absolute waste of time, complete garbage."

# Clean and vectorize the single string (Must use .transform, not .fit)
cleaned_review = clean_text_fast(custom_review)
vectorized_review = vectorizer.transform([cleaned_review])

# Predict (1 = Positive, 0 = Negative)
result = model.predict(vectorized_review)[0]
print(f"Prediction (1=Pos, 0=Neg): {result}")

Model Accuracy: 89.49%
Prediction (1=Pos, 0=Neg): 0
